In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from os import getenv
from pgvector_store import PGVectorStore
from ingest import load_faq_data, build_faiss_index
from rag_helper import RAGHelper

In [2]:
from sentence_transformers import SentenceTransformer

In [3]:
load_dotenv(override=True)

True

In [4]:
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
documents = load_faq_data()

In [7]:
class RAGPGVector(RAGHelper):
    def __init__(self, vector_store, **kwargs):
        super().__init__(index=None, **kwargs)
        self.vector_store = vector_store

    def search(self, query: str, num_results: int = 5) -> list[dict]:
        return self.vector_store.search(
            query=query,
            course=self.course,
            num_results=num_results,
        )

In [8]:
store = PGVectorStore(
    database_url=getenv("DATABASE_URL"),
    embedder=model,
)

store.ingest(documents)

vector_assistant = RAGPGVector(
    vector_store=store,
    llm_client=anthropic_client,
)

print(vector_assistant.rag("The program has already begun. Can I still sign up?"))

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Yes, you can still sign up! According to the course information:

- **You can join even after the program has begun**, as long as you want to participate in the learning materials.
- However, if you want to **receive a certificate, you need to submit your project while submissions are still being accepted**.

You don't even need to wait for a confirmation email to start — you can begin learning and submitting homework right away (as long as the submission form is still open). Registration is primarily used to gauge interest, not to gatekeep access to the course.
